In [2]:
import pandas as pd
import numpy as np
import os
import re

In [3]:
file_path = os.path.join('..', 'data', 'raw', 'train.csv')
df = pd.read_csv(file_path)

In [4]:
df.head()

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


In [5]:
import pandas as pd
import re
import numpy as np # Import numpy

# Assuming 'df' is your loaded DataFrame

# --- V3: A More Robust, Multi-Pattern Approach ---

def extract_item_size(text):
    """
    Extracts the primary item size (weight, volume) using a waterfall of regex patterns.
    """
    text = str(text)
    # Pattern 1: Look for numbers followed by specific units (oz, lb, g, ml, etc.)
    # This is the most common pattern. e.g., "16 oz", "1.5L", "500g"
    match = re.search(r'(\d+\.?\d*)\s*(oz|ounce|o z|lb|pound|g|gram|kg|kilo|ml|liter|l|fl oz|fluid ounce)', text, re.IGNORECASE)
    if match:
        return float(match.group(1))

    # Pattern 2: Look for hyphenated sizes, e.g., "1.9-Ounce"
    match = re.search(r'(\d+\.?\d*)-s*(?:oz|ounce|lb|g|kg|ml|l)', text, re.IGNORECASE)
    if match:
        return float(match.group(1))
        
    # Pattern 3: Look for numbers that are likely sizes but have no unit, often at the end of the string
    # This is a more general fallback. e.g., "The Original Butter Cookies, 8"
    match = re.search(r'(\d+\.?\d*)$', text.strip())
    if match:
        return float(match.group(1))

    return np.nan # Return NaN (Not a Number) if no size is found


def extract_pack_count(text):
    """
    Extracts the pack count using a waterfall of regex patterns.
    """
    text = str(text)
    # Pattern 1: Most specific - "Pack of X", "Case of X", "X-Count"
    # e.g., "Pack of 12", "20 Count", "4 ct", "6 pk", "12-pack"
    match = re.search(r'(?:Pack of|Pack|Case of|Case|Box of|Box|pk|ct|Count)\s*(\d+)|(\d+)\s*(?:-pack|-count|-ct)', text, re.IGNORECASE)
    if match:
        # The number can be in the first or second capture group
        return float(match.group(1) or match.group(2))

    # Pattern 2: Look for multiplication format, e.g., "2 x 14.1 oz" -> captures '2'
    match = re.search(r'(\d+)\s*x\s*\d+\.?\d*', text, re.IGNORECASE)
    if match:
        return float(match.group(1))
        
    return 1.0 # If no pack info is found, assume it's a single item (pack of 1)


# --- Apply the new, powerful functions ---
df['item_size'] = df['catalog_content'].apply(extract_item_size)
df['pack_count'] = df['catalog_content'].apply(extract_pack_count)

# --- Handle Missing Values and Create a Total Quantity Feature ---
# If item_size is still missing, we'll fill it with a neutral value (like the median or 1)
# For now, let's fill with 1. A more advanced step would be to use the median size.
df['item_size'].fillna(1.0, inplace=True) 

# This is a very powerful feature for the model!
df['total_quantity'] = df['item_size'] * df['pack_count']


# --- Check Your Work ---
# Display all the new columns to see how they performed
print("Comprehensive Feature Extraction Results:")
pd.set_option('display.max_rows', 50) # Show more rows for better inspection
print(df[['catalog_content', 'item_size', 'pack_count', 'total_quantity']].head(30))

# Finally, check how many item_sizes were missed (should be very few now)
print(f"\nTotal rows where item_size could not be determined: {df['item_size'].isna().sum()}")

Comprehensive Feature Extraction Results:
                                      catalog_content  item_size  pack_count  \
0   Item Name: La Victoria Green Taco Sauce Mild, ...      12.00         6.0   
1   Item Name: Salerno Cookies, The Original Butte...       8.00         4.0   
2   Item Name: Bear Creek Hearty Soup Bowl, Creamy...       1.90         6.0   
3   Item Name: Judee’s Blue Cheese Powder 11.25 oz...      11.25         1.0   
4   Item Name: kedem Sherry Cooking Wine, 12.7 Oun...      12.70         1.0   
5   Item Name: Member's Mark Member's Mark, Basil,...       6.25         1.0   
6   Item Name: Goya Foods Sazonador Total Seasonin...      30.00         6.0   
7   Item Name: VineCo Original Series Chilean Sauv...       8.00         1.0   
8   Item Name: NATURES PATH CEREAL FLK MULTIGRAIN ...      32.00         1.0   
9   Item Name: Mrs. Miller's Seedless Black Raspbe...       9.00         4.0   
10  Item Name: Braswell's Key Lime Marinade for So...      12.00         1.0  

C:\Users\nikk6\AppData\Local\Temp\ipykernel_18372\1419614779.py:61: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['item_size'].fillna(1.0, inplace=True)


In [6]:


# --- 2. Define the Comprehensive Feature Extraction Function (Your Code) ---

def find_quantity_and_unit(text):
    """
    Finds a quantity and its associated unit from a product title.
    Handles units that appear before or after the number.
    """
    # Define the units
    prefix_units = r'Pack of|Set of|Case of|Pack|Set'
    
    # All units that can appear AFTER a number
    postfix_units = (r'g|gm|grams|kg|kilograms|mg|milligrams|oz|ounce|ounces|lb|lbs|'
                     r'ml|milliliter|milliliters|l|ltr|liter|liters|cl|gal|gallon|qt|quart|'
                     r'fl oz|piece|pieces|pc|pcs|count|ct|unit|units|roll|rolls|pair|'
                     r'sheets|capsules|tablets|sachets|mm|cm|m|in|inch|inches|ft|feet|'
                     r'mah|gb|tb|mb|hz|ghz|mhz|w|watt|watts|v|volt|volts|amp|amps')

    # Regex to find a number followed by a unit OR a prefix unit followed by a number
    pattern = re.compile(
        # Pattern 1: Number followed by a unit (e.g., "500 g", "1.5 ltr")
        r'(?P<quantity1>\d*\.?\d+)\s*(?P<unit1>' + postfix_units + r')'
        r'|'  # OR
        # Pattern 2: Prefix unit followed by a number (e.g., "Pack of 6")
        r'(?P<unit2>' + prefix_units + r')\s*(?P<quantity2>\d+)',
        re.IGNORECASE
    )
    
    match = pattern.search(str(text))
    
    if match:
        # Check which pattern was matched and extract the correct groups
        if match.group('quantity1'):
            quantity = match.group('quantity1')
            unit = match.group('unit1')
        else:
            quantity = match.group('quantity2')
            unit = match.group('unit2')
        
        # Clean up and return
        return float(quantity), str(unit).strip().lower()
        
    return np.nan, None # Return NaN for quantity if nothing is found


# --- 3. Apply the Function and Create New Columns ---
# The .apply method returns a Series of tuples. We can expand this into a DataFrame.
temp_df = df['catalog_content'].apply(find_quantity_and_unit).apply(pd.Series)

# Assign the new columns to the main DataFrame
df['quantity'] = temp_df[0]
df['unit'] = temp_df[1]

# Handle missing values - fill quantity with a neutral value like 1
df['quantity'].fillna(1.0, inplace=True)
df['unit'].fillna('unit', inplace=True) # Fill missing units with a placeholder


# --- 4. Check Your Work ---
# Display all the new columns to see how they performed
print("Comprehensive Feature Extraction Results:")
pd.set_option('display.max_rows', 50)
print(df[['catalog_content', 'quantity', 'unit']].head(30))

Comprehensive Feature Extraction Results:
                                      catalog_content  quantity        unit
0   Item Name: La Victoria Green Taco Sauce Mild, ...     12.00       ounce
1   Item Name: Salerno Cookies, The Original Butte...      8.00       ounce
2   Item Name: Bear Creek Hearty Soup Bowl, Creamy...      1.90       ounce
3   Item Name: Judee’s Blue Cheese Powder 11.25 oz...     11.25          oz
4   Item Name: kedem Sherry Cooking Wine, 12.7 Oun...     12.70       ounce
5   Item Name: Member's Mark Member's Mark, Basil,...      6.25          oz
6   Item Name: Goya Foods Sazonador Total Seasonin...     30.00       ounce
7   Item Name: VineCo Original Series Chilean Sauv...      8.00           l
8   Item Name: NATURES PATH CEREAL FLK MULTIGRAIN ...     32.00          oz
9   Item Name: Mrs. Miller's Seedless Black Raspbe...      9.00       ounce
10  Item Name: Braswell's Key Lime Marinade for So...     12.00          oz
11  Item Name: Albanese Assorted Gummi Bears, 

C:\Users\nikk6\AppData\Local\Temp\ipykernel_18372\470963425.py:54: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['quantity'].fillna(1.0, inplace=True)
C:\Users\nikk6\AppData\Local\Temp\ipykernel_18372\470963425.py:55: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, wh

In [7]:
import re

# small word->number map
_WORD_NUM_MAP = {
    "half dozen": 6,
    "dozen": 12,
    "single": 1,
    "pair": 2,
    "double": 2,
    "triple": 3,
    "trio": 3,
}

def _to_float_safe(s):
    try:
        return float(s)
    except Exception:
        return None

def extract_pack_count(text):
    """
    Extract pack count from product title/description text.
    Returns a float (pack count) or 1.0 if nothing reliable found.
    Heuristics covered: "Pack of 12", "12-pack", "4pk", "3 x 2" (=> 6),
    "2-in-1" (=>2), "2 for 1" (=>2), "dozen", "pair", "pcs/pieces/ct", parentheses like "(Pack of 4)",
    and a fallback choosing the largest reasonable integer (2..500) in text.
    """
    if text is None:
        return 1.0

    s = str(text).lower()
    s = s.replace('×', 'x').replace('X', 'x').replace(',', '')
    s = re.sub(r'[^\x00-\x7f]', ' ', s)  # normalize weird unicode

    # 1) word map
    for k, v in _WORD_NUM_MAP.items():
        if k in s:
            return float(v)

    # 2) explicit parenthetical "(Pack of 6)" etc
    m = re.search(r'\((?:pack(?: of)?|pk|pks|packs|count|ct)\s*(\d{1,4})\)', s, re.IGNORECASE)
    if m:
        val = _to_float_safe(m.group(1))
        if val and val >= 1:
            return float(val)

    # 3) explicit phrases "pack of 12", "case of 6", "box of 4"
    m = re.search(r'\b(?:pack(?:age)?|case|box|set|bundle|combo|multi-?pack|value ?pack|packet|packets|count|ct)\b[^\d]{0,6}?(\d{1,4}(?:\.\d+)?)', s, re.IGNORECASE)
    if m:
        val = _to_float_safe(m.group(1))
        if val and val >= 1:
            return float(val)

    # 4) suffix/prefix forms: "12-pack", "12 pack", "12pk", "6ct", "12pcs"
    m = re.search(r'\b(\d{1,4}(?:\.\d+)?)(?:\s|-)?(?:pack|pk|pks|packs|pkg|ct|count|pcs?|pieces?|units?|unit|btl|bottles|rolls|bars|pairs)\b', s, re.IGNORECASE)
    if m:
        val = _to_float_safe(m.group(1))
        if val and val >= 1:
            return float(val)

    # 5) unit words after number "4 pcs", "3 pieces", "12 units"
    m = re.search(r'\b(\d{1,4}(?:\.\d+)?)\s*(?:pcs?|pieces?|units?|bottles?|rolls?|bars?|pairs?)\b', s, re.IGNORECASE)
    if m:
        val = _to_float_safe(m.group(1))
        if val and val >= 1:
            return float(val)

    # 6) multiplicative forms: "3 x 2", "3x2" -> treat as product (3*2 = 6)
    m = re.search(r'\b(\d{1,3}(?:\.\d+)?)\s*[x\*]\s*(\d{1,3}(?:\.\d+)?)\b', s, re.IGNORECASE)
    if m:
        g1 = _to_float_safe(m.group(1))
        g2 = _to_float_safe(m.group(2))
        if g1 and g2:
            # Common interpretation for titles: 3 x 2 => 3 packs of 2 => total 6
            prod = g1 * g2
            if prod >= 1:
                return float(prod)

    # 7) "2-in-1" or "2 in 1" => return first number (2)
    m = re.search(r'\b(\d{1,3})\s*(?:-in-|\s+in\s+)\s*(\d{1,3})\b', s, re.IGNORECASE)
    if m:
        g1 = _to_float_safe(m.group(1))
        if g1 and g1 >= 1:
            return float(g1)

    # 8) "2 for 1" -> return first number if second==1 (promo)
    m = re.search(r'\b(\d{1,3})\s*(?:for|/)\s*(\d{1,3})\b', s, re.IGNORECASE)
    if m:
        g1 = int(m.group(1)); g2 = int(m.group(2))
        if g1 >= 1 and g2 == 1:
            return float(g1)
        # if it's like "3 for 2", return first number as heuristic
        if g1 >= 1:
            return float(g1)

    # 9) fallback: pick largest reasonable integer found (2..500)
    nums = [int(n) for n in re.findall(r'\b(\d{1,4})\b', s)]
    nums = [n for n in nums if 2 <= n <= 500]
    if nums:
        return float(max(nums))

    # final default
    return 1.0


In [8]:
df['pack_count'] = df['catalog_content'].apply(extract_pack_count)

In [9]:
df.head(10)

,sample_id,catalog_content,image_link,price,item_size,pack_count,total_quantity,quantity,unit
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,12.00,6.00,72.00,12.00,ounce
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,8.00,4.00,32.00,8.00,ounce
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,1.90,1.00,11.40,1.90,ounce
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,11.25,11.25,11.25,11.25,oz
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,12.70,12.00,12.70,12.70,ounce
5,9259,"Item Name: Member's Mark Member's Mark, Basil,...",https://m.media-amazon.com/images/I/81nw0HXpCR...,18.50,6.25,6.25,6.25,6.25,oz
6,191846,Item Name: Goya Foods Sazonador Total Seasonin...,https://m.media-amazon.com/images/I/61dH2Ebkt0...,5.99,30.00,6.00,180.00,30.00,ounce
7,222007,Item Name: VineCo Original Series Chilean Sauv...,https://m.media-amazon.com/images/I/71JllaFpxM...,94.00,8.00,2.00,8.00,8.00,l
8,37614,Item Name: NATURES PATH CEREAL FLK MULTIGRAIN ...,https://m.media-amazon.com/images/I/21O9RftI2v...,35.74,32.00,192.00,32.00,32.00,oz
9,238044,Item Name: Mrs. Miller's Seedless Black Raspbe...,https://m.media-amazon.com/images/I/41miQk+RkJ...,31.80,9.00,4.00,36.00,9.00,ounce


In [10]:
# --- 5. Save the Processed Training Data ---

# Define the path for the output file
# The '../' correctly navigates up from the 'notebooks' folder to the project root
train_processed_path = os.path.join('..', 'data', 'processed', 'train_features_text.csv')

# Save the DataFrame to a CSV file
print(f"Saving processed training data to: {train_processed_path}")
df.to_csv(train_processed_path, index=False)

print("✅ Training data saved successfully.")

Saving processed training data to: ..\data\processed\train_features_text.csv
✅ Training data saved successfully.


In [12]:
# --- 6. Process and Save the Test Data ---

print("Processing test data...")

# --- Load the raw test data ---
test_raw_path = os.path.join('..', 'data', 'raw', 'test.csv')
test_df = pd.read_csv(test_raw_path)

# --- Apply the EXACT SAME feature engineering functions from the cells above ---
temp_test_df = test_df['catalog_content'].apply(find_quantity_and_unit).apply(pd.Series)
test_df['item_size'] = temp_test_df[0]
test_df['unit'] = temp_test_df[1]

test_df['pack_count'] = test_df['catalog_content'].apply(extract_pack_count)

# --- Clean up and create the final feature ---
test_df['item_size'].fillna(1.0, inplace=True)
test_df['unit'].fillna('unit', inplace=True)
test_df['total_quantity'] = test_df['item_size'] * test_df['pack_count']

# --- Save the processed test data ---
test_processed_path = os.path.join('..', 'data', 'processed', 'test_features_text.csv')
print(f"Saving processed test data to: {test_processed_path}")
test_df.to_csv(test_processed_path, index=False)

print("✅ Test data saved successfully.")
print(test_df[['catalog_content', 'item_size', 'pack_count']].head())

Processing test data...


C:\Users\nikk6\AppData\Local\Temp\ipykernel_18372\3786809701.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df['item_size'].fillna(1.0, inplace=True)
C:\Users\nikk6\AppData\Local\Temp\ipykernel_18372\3786809701.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

Saving processed test data to: ..\data\processed\test_features_text.csv
✅ Test data saved successfully.
                                     catalog_content  item_size  pack_count
0  Item Name: Rani 14-Spice Eshamaya's Mango Chut...       10.5        10.5
1  Item Name: Natural MILK TEA Flavoring extract ...        2.0         1.0
2  Item Name: Honey Filled Hard Candy - Bulk Pack...        2.0         2.0
3  Item Name: Vlasic Snack'mm's Kosher Dill 16 Oz...       16.0         2.0
4  Item Name: McCormick Culinary Vanilla Extract,...       32.0        32.0
